# Stage 2 — Model Zoo (raw, untrained two-sided models)

Each architecture in the zoo is **instantiated untrained** (random weights,
fixed seed) and immediately saved to `models/raw/<arch>/` as two files:

| file | contents |
|------|----------|
| `model.pt` | `state_dict` — raw (untrained) weights via `torch.save` |
| `arch.json` | constructor recipe `{class, kwargs, variant}` |

Downstream notebooks (**train_and_test**, **comparison**) load
`arch.json`, reconstruct the model with `build_from_arch`, then call
`load_state_dict` to restore weights — they never hard-code constructor args.
This single source of truth means adding a new variant here automatically
propagates to training and comparison.


In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath('../src'))
import torch
import csi
from pathlib import Path

print('torch', torch.__version__)
print('RAW_ROOT', csi.RAW_ROOT)

torch 2.11.0
RAW_ROOT /Users/hjl/Documents/Github/csi_report/models/raw


In [2]:
# ── Model Zoo ────────────────────────────────────────────────────────────
# Each entry: (variant_name, class_name, constructor kwargs)
# n_code = M = latent / feedback dimension. Two model families at matched M:
#   CsiNet   — the original CNN autoencoder (legacy baseline)
#   TransNet — transformer (full-attention) codec (the replacement model)
ZOO = [
    ('csinet16',    'CsiNet',   dict(n_delay=32, n_tx=32, n_code=16, final_activation='linear')),
    ('csinet32',    'CsiNet',   dict(n_delay=32, n_tx=32, n_code=32, final_activation='linear')),
    ('csinet64',    'CsiNet',   dict(n_delay=32, n_tx=32, n_code=64, final_activation='linear')),
    ('transnet16',  'TransNet', dict(n_delay=32, n_tx=32, n_code=16, final_activation='linear')),
    ('transnet32',  'TransNet', dict(n_delay=32, n_tx=32, n_code=32, final_activation='linear')),
    ('transnet64',  'TransNet', dict(n_delay=32, n_tx=32, n_code=64, final_activation='linear')),
]

print('Zoo entries:', [name for name, _, _ in ZOO])

Zoo entries: ['csinet16', 'csinet32', 'csinet64', 'transnet16', 'transnet32', 'transnet64']


In [3]:
# ── Shared loader helper ─────────────────────────────────────────────────
# train_and_test and comparison notebooks use the IDENTICAL rule:
#   model = build_from_arch(json.loads((d / 'arch.json').read_text()))
# Define it once here; copy-paste (or import) into downstream notebooks.

def build_from_arch(arch: dict):
    # arch = {'class': 'CsiNet', 'kwargs': {...}, 'variant': '...'}
    cls = getattr(csi, arch['class'])
    return cls(**arch['kwargs'])

print('build_from_arch defined — train/comparison use the identical rule')

build_from_arch defined — train/comparison use the identical rule


In [4]:
# ── Build & save raw (untrained) models + record complexity ──────────────
for name, cls, kw in ZOO:
    torch.manual_seed(0)                          # reproducible random init
    net = getattr(csi, cls)(**kw)

    d = Path(csi.RAW_ROOT) / name
    d.mkdir(parents=True, exist_ok=True)

    # weights
    torch.save(net.state_dict(), d / 'model.pt')

    # complexity (params exact, FLOPs estimated) — saved into arch.json
    cx = csi.model_complexity(net, input_shape=(1, 2, kw['n_delay'], kw['n_tx']))

    # architecture recipe (+ complexity for the comparison report)
    arch = {'class': cls, 'kwargs': kw, 'variant': name,
            'params': cx['params'], 'flops': cx['flops']}
    (d / 'arch.json').write_text(json.dumps(arch, indent=2))

    print(f'saved raw  {name:<12} {cls:<9} params={cx["params"]:>10,}  '
          f'flops={cx["flops"]/1e6:7.2f} MFLOP  ->  {d}')

saved raw  csinet16     CsiNet    params=    71,592  flops=   7.87 MFLOP  ->  /Users/hjl/Documents/Github/csi_report/models/raw/csinet16
saved raw  csinet32     CsiNet    params=   137,144  flops=   8.00 MFLOP  ->  /Users/hjl/Documents/Github/csi_report/models/raw/csinet32
saved raw  csinet64     CsiNet    params=   268,248  flops=   8.27 MFLOP  ->  /Users/hjl/Documents/Github/csi_report/models/raw/csinet64
saved raw  transnet16   TransNet  params=   953,040  flops=  49.55 MFLOP  ->  /Users/hjl/Documents/Github/csi_report/models/raw/transnet16
saved raw  transnet32   TransNet  params= 1,084,128  flops=  49.81 MFLOP  ->  /Users/hjl/Documents/Github/csi_report/models/raw/transnet32
saved raw  transnet64   TransNet  params= 1,346,304  flops=  50.33 MFLOP  ->  /Users/hjl/Documents/Github/csi_report/models/raw/transnet64


In [5]:
# ── Verification: reload arch.json, rebuild, load weights ────────────────
for name, _, _ in ZOO:
    d = Path(csi.RAW_ROOT) / name

    # reconstruct from recipe (same rule downstream notebooks use)
    arch = json.loads((d / 'arch.json').read_text())
    net = build_from_arch(arch)

    # load saved weights
    m = net.load_state_dict(torch.load(d / 'model.pt', weights_only=True))
    assert not m.missing_keys,    f'{name}: missing keys {m.missing_keys}'
    assert not m.unexpected_keys, f'{name}: unexpected keys {m.unexpected_keys}'

print('ZOO OK — all variants round-trip cleanly')

ZOO OK — all variants round-trip cleanly


## arch.json contract

```json
{
  "class":   "CsiNet",
  "kwargs":  { "n_delay": 32, "n_tx": 32, "n_code": 64, "final_activation": "linear" },
  "variant": "csinet64"
}
```

**Fields:**

| field | meaning |
|-------|---------|
| `class` | Attribute name on the `csi` module — passed to `getattr(csi, arch['class'])` |
| `kwargs` | Constructor keyword arguments, unpacked verbatim |
| `variant` | Human-readable name; also used as the subdirectory under `models/raw/` |

Both **train_and_test** and **comparison** load this schema via
`build_from_arch(arch)` — no hard-coded constructor calls outside this zoo.
